In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
cd /content
rm -rf pcb-defect
git clone --depth 1 https://github.com/akshatyuvan/pcb-defect.git
cd pcb-defect
rm -rf data artifacts
ln -s /content/drive/MyDrive/pcb-defect/data      data
ln -s /content/drive/MyDrive/pcb-defect/artifacts artifacts
ls -la | grep -E ' data| artifacts'
ls -la artifacts/ | head
git log --oneline -1

In [ ]:
%%bash
cd /content/pcb-defect
ls -l artifacts/*.pt
md5sum artifacts/best.pt artifacts/r4_weighted_p05_registered_best.pt
cat artifacts/classes.json
cat artifacts/norm.json

In [ ]:
import sys; sys.path.insert(0, '/content/pcb-defect')
import os; os.chdir('/content/pcb-defect')
from src.models.checkpoint import load_pcbnet, load_classes

classes = load_classes('artifacts')
model = load_pcbnet('artifacts/r4_weighted_p05_registered_best.pt', len(classes), 'cpu')

for name, mod in model.named_modules():
    if name and '.' not in name.split('.', 1)[-1][:0] and len(name.split('.')) <= 2:
        print(f'{name:<28} {type(mod).__name__}')

In [ ]:
import torch
from src.models.checkpoint import load_pcbnet, load_classes

# reload fresh — the previous model object has broken hooks stuck on it
classes = load_classes('artifacts')
model = load_pcbnet('artifacts/r4_weighted_p05_registered_best.pt', len(classes), 'cpu')

x = torch.zeros(1, 1, 64, 64)
shapes = {}

def make_hook(name):
    def hook(m, i, o):
        t = o[0] if isinstance(o, (tuple, list)) else o
        if hasattr(t, "shape"):
            shapes[name] = tuple(t.shape)
    return hook

hooks = [m.register_forward_hook(make_hook(n)) for n, m in model.named_modules() if n]

try:
    with torch.no_grad():
        model(x)
finally:
    for h in hooks:
        h.remove()   # guaranteed to run even if the forward pass errors

for n, s in shapes.items():
    if len(s) == 4 and s[-1] in (4, 8, 16):
        print(f'{n:<30} {s}')

In [ ]:
%%bash
cd /content/pcb-defect
python -m src.explain.gradcam \
  --patches data/patches \
  --raw data/raw/PCBData \
  --artifacts artifacts \
  --ckpt artifacts/r4_weighted_p05_registered_best.pt \
  --layers final,features.2 \
  --boards 2 \
  --tag day4_gradcam 2>&1 | tail -100

In [ ]:
print(open('artifacts/day4_gradcam_report.txt').read())

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('artifacts/figures/day4_*.png')):
    print(p); display(Image(p))

In [ ]:
%%bash
cd /content/pcb-defect/artifacts
zip -r /content/drive/MyDrive/pcb-defect/day4_artifacts.zip \
  day4_gradcam_report.txt day4_gradcam_summary.json figures/day4_*.png
ls -lh /content/drive/MyDrive/pcb-defect/day4_artifacts.zip